## Analyze the bins

In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

In [2]:
# --- CONFIGURATION ---
BASE_DIR = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data"
SPLIT_DIR = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits"

ORIGINAL_TRAIN = f"{SPLIT_DIR}/stratified_binary_07_dataset_train.feather"
BIG_MIX_SOURCE = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/mass_gate_2_percent/temp_subset_0.02_stratified_binary_07_spec_sim_dataset_train.feather" # REPLACE THIS

# Metadata
COMBINED_MOL = f"{BASE_DIR}/mol_df_spec_sim.pkl"
COMBINED_SPEC = f"{BASE_DIR}/spec_df_spec_sim.pkl"
VAL_PATH = f"{SPLIT_DIR}/stratified_binary_07_dataset_val.feather"
TEST_PATH = f"{SPLIT_DIR}/stratified_binary_07_dataset_test.feather"

OUTPUT_PATH = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/mass_gate_2_percent/stratified_binary_07_dataset_train_ADAPTIVE_GRID_large.feather"

In [6]:
# Bounds for the Adaptive Target
MIN_TARGET = 2000  # Never go below this (unless data literally doesn't exist)
MAX_TARGET = 7000  # Never go above this (to prevent memory explosion)

SIM_BINS = [
    (0.0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0001)
]

In [7]:
# --- 1. SETUP FIREWALL & HELPERS ---
print("--- Initializing ---")
df_spec = pd.read_pickle(COMBINED_SPEC)
df_mol = pd.read_pickle(COMBINED_MOL)
mol2scaffold = dict(zip(df_mol['mol_id'], df_mol['scaffold']))
spec2mol = dict(zip(df_spec['spec_id'], df_spec['mol_id']))

--- Initializing ---


In [8]:
def get_scaffold(spec_id):
    mol_id = spec2mol.get(spec_id)
    if mol_id is None: return "Unknown"
    return mol2scaffold.get(mol_id, "Unknown")

# Firewall
df_val = pd.read_feather(VAL_PATH)
df_test = pd.read_feather(TEST_PATH)
forbidden_specs = set(df_val['name_main']) | set(df_val['name_sub']) | \
                  set(df_test['name_main']) | set(df_test['name_sub'])
forbidden_scaffolds = set()
for spec_id in tqdm(forbidden_specs, desc="Building Firewall"):
    s = get_scaffold(spec_id)
    if s != "Unknown": forbidden_scaffolds.add(s)

Building Firewall: 100%|██████████| 6166/6166 [00:00<00:00, 412440.45it/s]


In [9]:
# --- 2. LOAD & PREPROCESS ---
print("\n--- Loading Data ---")
df_orig = pd.read_feather(ORIGINAL_TRAIN)
df_mix = pd.read_feather(BIG_MIX_SOURCE) 

if 'label' not in df_mix.columns:
    df_mix['label'] = (df_mix['cosine_similarity'] >= 0.7).astype(int)

def get_regime(row):
    md = row.get('mass_difference', -1)
    if md < 0: return "Unknown"
    if md < 0.01: return "0. Exact Isomers"
    if md < 1.0: return "1. Isobaric"
    if md < 10.0: return "2. Tiny (<10Da)"
    if md < 50.0: return "3. Medium (<50Da)"
    if md < 100.0: return "4. Large (<100Da)"
    return "5. Huge (>100Da)"

def get_sim_bin(sim):
    for i, (low, high) in enumerate(SIM_BINS):
        if low <= sim < high: return i
    return 4

print("Preprocessing Original...")
if 'mass_difference' not in df_orig.columns:
    spec2mz = dict(zip(df_spec['spec_id'], df_spec['prec_mz']))
    df_orig['mass_difference'] = abs(df_orig['name_main'].map(spec2mz) - df_orig['name_sub'].map(spec2mz))
df_orig['Regime_Norm'] = df_orig.apply(get_regime, axis=1)
df_orig['Sim_Bin_Idx'] = df_orig['cosine_similarity'].apply(get_sim_bin)

print("Preprocessing Mixed...")
if 'mass_difference' not in df_mix.columns:
    spec2mz = dict(zip(df_spec['spec_id'], df_spec['prec_mz']))
    df_mix['mass_difference'] = abs(df_mix['name_main'].map(spec2mz) - df_mix['name_sub'].map(spec2mz))
df_mix['Regime_Norm'] = df_mix.apply(get_regime, axis=1)
df_mix['Sim_Bin_Idx'] = df_mix['cosine_similarity'].apply(get_sim_bin)


--- Loading Data ---
Preprocessing Original...
Preprocessing Mixed...


In [10]:
# --- 3. DIVERSITY SAMPLER ---
def sample_diverse(df_subset, n):
    if len(df_subset) <= n: return df_subset
    df_subset = df_subset.copy()
    df_subset['scaff'] = [get_scaffold(x) for x in df_subset['name_main']]
    groups = df_subset.groupby('scaff')
    keys = list(groups.groups.keys())
    np.random.shuffle(keys)
    indices = []
    iters = [iter(groups.get_group(k).index) for k in keys]
    while len(indices) < n:
        active = []
        for it in iters:
            try:
                indices.append(next(it))
                active.append(it)
                if len(indices) >= n: break
            except StopIteration: pass
        iters = active
        if not iters: break
    return df_subset.loc[indices].drop(columns=['scaff'])

In [11]:
# --- 4. EXECUTE ADAPTIVE EQUALIZER ---
final_dfs = []
regimes = sorted(df_orig['Regime_Norm'].unique())

print(f"\n--- Starting Adaptive Equalization ---")

for regime in regimes:
    print(f"\n=== Calculating Target for {regime} ===")
    
    # 1. Scan Availability in Mixed Source (Just Counts)
    # This assumes Mixed Source is superset or we can pull freely from it
    mix_counts = df_mix[df_mix['Regime_Norm'] == regime]['Sim_Bin_Idx'].value_counts()
    orig_counts = df_orig[df_orig['Regime_Norm'] == regime]['Sim_Bin_Idx'].value_counts()
    
    # Estimate total available 'Safe' pairs (Heuristic: 50% of Mix is usually safe)
    # We take the count of the 3rd largest bin (Median-ish) to set the bar
    total_avail_per_bin = [orig_counts.get(i,0) + (mix_counts.get(i,0) * 0.5) for i in range(5)]
    estimated_capacity = np.median(total_avail_per_bin)
    
    # Clamp the Target
    REGIME_TARGET = int(max(MIN_TARGET, min(estimated_capacity, MAX_TARGET)))
    print(f"   > Estimated Capacity: {int(estimated_capacity)}")
    print(f"   > Set Adaptive Target: {REGIME_TARGET}")
    
    # 2. Execute Fill
    for bin_idx, (low, high) in enumerate(SIM_BINS):
        subset_orig = df_orig[
            (df_orig['Regime_Norm'] == regime) & 
            (df_orig['Sim_Bin_Idx'] == bin_idx)
        ]
        count_orig = len(subset_orig)
        deficit = REGIME_TARGET - count_orig
        
        print(f"     [Sim {low}-{high:.1f}] Orig: {count_orig}", end="")
        
        # CASE A: Downsample
        if deficit < 0:
            print(" -> Downsampling")
            final_dfs.append(sample_diverse(subset_orig, REGIME_TARGET))
            
        # CASE B: Fill
        else:
            print(f" -> Filling {deficit}", end="")
            final_dfs.append(subset_orig)
            
            if deficit > 0:
                candidates = df_mix[
                    (df_mix['Regime_Norm'] == regime) & 
                    (df_mix['Sim_Bin_Idx'] == bin_idx)
                ]
                
                # JIT Firewall Check
                candidates = candidates.sample(frac=1.0, random_state=42)
                safe_rows = []
                for idx, row in candidates.iterrows():
                    if len(safe_rows) >= deficit: break
                    s1 = get_scaffold(row['name_main'])
                    s2 = get_scaffold(row['name_sub'])
                    if s1 not in forbidden_scaffolds and s2 not in forbidden_scaffolds:
                        safe_rows.append(row)
                        
                if safe_rows:
                    added = pd.DataFrame(safe_rows)
                    cols = list(set(df_orig.columns) & set(added.columns))
                    final_dfs.append(added[cols])
                    print(f" (Added {len(added)})")
                else:
                    print(" (No safe candidates)")
            else:
                print("")


--- Starting Adaptive Equalization ---

=== Calculating Target for 0. Exact Isomers ===
   > Estimated Capacity: 1110
   > Set Adaptive Target: 2000
     [Sim 0.0-0.2] Orig: 27 -> Filling 1973 (Added 220)
     [Sim 0.2-0.4] Orig: 138 -> Filling 1862 (Added 531)
     [Sim 0.4-0.6] Orig: 436 -> Filling 1564 (Added 1187)
     [Sim 0.6-0.8] Orig: 2250 -> Downsampling
     [Sim 0.8-1.0] Orig: 7451 -> Downsampling

=== Calculating Target for 1. Isobaric ===
   > Estimated Capacity: 1291
   > Set Adaptive Target: 2000
     [Sim 0.0-0.2] Orig: 147 -> Filling 1853 (Added 994)
     [Sim 0.2-0.4] Orig: 219 -> Filling 1781 (Added 1383)
     [Sim 0.4-0.6] Orig: 285 -> Filling 1715 (Added 1672)
     [Sim 0.6-0.8] Orig: 642 -> Filling 1358 (Added 1358)
     [Sim 0.8-1.0] Orig: 910 -> Filling 1090 (Added 1090)

=== Calculating Target for 2. Tiny (<10Da) ===
   > Estimated Capacity: 10075
   > Set Adaptive Target: 7000
     [Sim 0.0-0.2] Orig: 1306 -> Filling 5694 (Added 5694)
     [Sim 0.2-0.4] Orig:

In [12]:
# --- 5. SAVE ---
print("\n--- Finalizing ---")
df_final = pd.concat(final_dfs, axis=0, ignore_index=True)
df_final = df_final.sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f"Total Size: {len(df_final)}")
df_final.to_feather(OUTPUT_PATH)
print(f"Saved to: {OUTPUT_PATH}")


--- Finalizing ---
Total Size: 155239
Saved to: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/mass_gate_2_percent/stratified_binary_07_dataset_train_ADAPTIVE_GRID_large.feather
